In [1]:
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import pandas as pd
import os
import tensorflow as tf
from tensorflow import keras
import pickle
import numpy.random as rand
import h5py
import time
from tqdm import tqdm
from pathlib import Path

In [ ]:
# 경로 설정: 이 저장소(PDSI)를 어디에 클론해도 동작하도록, 위쪽 폴더에서
# "PDSI"라는 이름의 git 저장소를 찾아 그 형제 폴더인 "DATA"를 데이터 루트로 씁니다.
# DATA 폴더 위치가 다르면 환경변수 PDSI_DATA_ROOT 로 직접 지정하세요
# (DATA 폴더를 담고 있는 상위 폴더 경로를 넣으면 됩니다).
def _find_project_root(marker="PDSI"):
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if candidate.name == marker and (candidate / ".git").exists():
            return candidate
    return None

_env_override = os.environ.get("PDSI_DATA_ROOT")
if _env_override:
    base = Path(_env_override)
else:
    _root = _find_project_root()
    if _root is None:
        raise FileNotFoundError(
            "'PDSI' 저장소 폴더를 상위 경로에서 찾지 못했습니다. "
            "환경변수 PDSI_DATA_ROOT 에 DATA 폴더의 상위 경로를 직접 지정하세요."
        )
    base = _root.parent
print(f"데이터 루트(base): {base}")
model_dir = [base / "DATA" / "CA_77" / "models" / "model_1.keras", 
             base / "DATA" / "CA_77" / "models" / "model_2.keras", 
             base / "DATA" / "CA_77" / "models" / "model_3.keras", 
             base / "DATA" / "CA_77" / "models" / "model_4.keras", 
             base / "DATA" / "CA_77" / "models" / "model_5.keras"]

loaded_models = [keras.models.load_model(d) for d in model_dir]  # 모델 5개를 한 번만 로드
local_index = pd.read_csv(base / "DATA" / "SDM_data" / "latin" / "local_index.csv")
local_name = local_index["SIG_ENG_NM"].tolist()

with open(base / "DATA" / "SDM_data" / "latin" / "TES_maxent" / "all.pkl", 'rb') as file:
    local_all = pickle.load(file)
with open(base / "DATA" / "SDM_data" / "latin" / "TES_maxent" / "sampling.pkl", 'rb') as file:
    local_lhs = pickle.load(file)
species_path = str(base / "DATA" / "SDM_data" / "latin" / "TES_maxent") #분석하고 싶은 종 선택 (TES)
scenario = ['126', '126', '126', '126'] #시나리오 선택
seeed_num = 10

In [3]:
#lhs
def make_maxent_mat(scenario, loc):
    maxent_mat = []
    for i in ["ssp"+str(scenario[0])+"_2030", "ssp"+str(scenario[1])+"_2050",
              "ssp"+str(scenario[2])+"_2070", "ssp"+str(scenario[3])+"_2090"]:
        dataF = local_lhs[i][loc]
        dataF = dataF.iloc[:,2]
        dataF = np.reshape(dataF,(20,20))
        maxent_mat.append(dataF)
        
    maxent_mat = np.array(maxent_mat)
    maxent_mat = np.transpose(maxent_mat,(1,2,0))
    maxent_mat = np.reshape(maxent_mat,(1,20,20,4))
    return maxent_mat

In [4]:
labels = [8, 6, 2, 34, 38, 12, 10, 14, 26, 30, 58, 62, 74, 78, 106, 110, 40, 44, 42, 46, 136, 140, 138, 142, 154, 158, 168, 172, 170, 174, 186, 190, 130, 134, 162, 166, 234, 238, 202, 206, 24, 28, 56, 60, 152, 156, 184, 188, 4, 18, 22, 36, 50, 54, 72, 76, 90, 94, 104, 108, 122, 126, 132, 146, 150, 160, 164, 178, 182, 200, 204, 218, 222, 232, 236, 250, 254]

In [5]:
weight_df = pd.read_csv(base / "DATA" / "weights" / "WeightByInitial_new.csv")
weight = weight_df.pivot(index="rule", columns="initial", values="new_mean_gen60")  # weight.loc[rule, initial] == 기존 weight[rule][initial]

In [6]:
def clu_SI(initial, CA_distribution):
    global weight, labels
    SI = 0
    for k, i in enumerate(labels):  # k는 인덱스, i는 labels의 값
        si = CA_distribution[0][k] * weight.loc[i, initial]  # 각 원소를 선택하여 저장
        SI += si
        
    # # 전체 면적으로 나눌 때    
    SI = SI/400
    # # 초기갑으로 나눌 때
    # if initial != 0:
    #     SI = SI / initial
    # else:
    #     SI = 0
    
    return SI

In [7]:
scenarios = [['126','126','126','126'],['126','126','126','245'],['126','126','126','585'],['126','126','245','126'],['126','126','245','245'],['126','126','245','585'],['126','126','585','126'],['126','126','585','245'],['126','126','585','585'],
             ['126','245','126','126'],['126','245','126','245'],['126','245','126','585'],['126','245','245','126'],['126','245','245','245'],['126','245','245','585'],['126','245','585','126'],['126','245','585','245'],['126','245','585','585'],
             ['126','585','126','126'],['126','585','126','245'],['126','585','126','585'],['126','585','245','126'],['126','585','245','245'],['126','585','245','585'],['126','585','585','126'],['126','585','585','245'],['126','585','585','585'],
             ['245','126','126','126'],['245','126','126','245'],['245','126','126','585'],['245','126','245','126'],['245','126','245','245'],['245','126','245','585'],['245','126','585','126'],['245','126','585','245'],['245','126','585','585'],
             ['245','245','126','126'],['245','245','126','245'],['245','245','126','585'],['245','245','245','126'],['245','245','245','245'],['245','245','245','585'],['245','245','585','126'],['245','245','585','245'],['245','245','585','585'],
             ['245','585','126','126'],['245','585','126','245'],['245','585','126','585'],['245','585','245','126'],['245','585','245','245'],['245','585','245','585'],['245','585','585','126'],['245','585','585','245'],['245','585','585','585'],
             ['585','126','126','126'],['585','126','126','245'],['585','126','126','585'],['585','126','245','126'],['585','126','245','245'],['585','126','245','585'],['585','126','585','126'],['585','126','585','245'],['585','126','585','585'],
             ['585','245','126','126'],['585','245','126','245'],['585','245','126','585'],['585','245','245','126'],['585','245','245','245'],['585','245','245','585'],['585','245','585','126'],['585','245','585','245'],['585','245','585','585'],
             ['585','585','126','126'],['585','585','126','245'],['585','585','126','585'],['585','585','245','126'],['585','585','245','245'],['585','585','245','585'],['585','585','585','126'],['585','585','585','245'],['585','585','585','585']]

In [8]:
# 34개 지역 중 기존 19개에 없던 15개 지역만 계산 (나머지 19개는 이미 계산 완료)
regions = ["Ansan-si", "Anyang-si", "Bucheon-si", "Goyang-si", "Gunpo-si", "Guri-si", "Gwacheon-si", "Gwangmyeong-si", "Hanam-si", "Osan-si", "Seongnam-si", "Seoul-si", "Siheung-si", "Suwon-si", "Uiwang-si"]
n_rep = 100
assert len(scenarios) == 81 and all(r in local_name for r in regions)
res_dir = base / "DATA" / "Results"
names = ["_".join(s) for s in scenarios]
out_cellwise = res_dir / "77_lhs_repeat100_type1_newweight_extra15.csv"
out_global = res_dir / "77_lhs_repeat100_type2_newweight_extra15.csv"

In [9]:
# 기존 방식: 격자칸마다 난수를 따로 뽑아 바이너리화 (새 weight 로 SI 계산)
def make_CA_distribution_cellwise(maxent_mat, model):
    b = np.zeros((1, int(model)), dtype=np.float32); initial = 0
    for model_call in loaded_models:
        binary_batch = (np.random.rand(100, 20, 20, 4) < maxent_mat).astype(np.float32)
        initial += np.sum(binary_batch[:,:,:,0])
        b += np.sum(np.array(model_call(binary_batch), dtype=np.float32), axis=0, keepdims=True)
    return int(initial/500), b/500

In [10]:
# 새 방식(global): 위치(i,j)별 난수 R_ij 400개를 뽑아 4개 시기(2030~2090)의 같은 위치 칸이 공유.
# 반복 번호 rep 마다 시드를 고정해 500세트를 한 번에 뽑고(서로 모두 다름), 100개씩 5조각으로 모델 5개에 나눠 넣는다.
# 같은 rep 는 모든 지역/시나리오에서 같은 R_ij 를 쓴다 (저장 없이 재현).
def make_CA_distribution_global(maxent_mat, model, rep):
    R_all = np.random.default_rng(rep).random((500, 20, 20, 1))
    b = np.zeros((1, int(model)), dtype=np.float32); initial = 0
    for m, model_call in enumerate(loaded_models):
        R = R_all[m*100:(m+1)*100]
        binary_batch = (R < maxent_mat).astype(np.float32)
        initial += np.sum(binary_batch[:,:,:,0])
        b += np.sum(np.array(model_call(binary_batch), dtype=np.float32), axis=0, keepdims=True)
    return int(initial/500), b/500

In [11]:
# 시나리오 x 지역 마다 cell-wise, global 두 방식을 100회씩 계산 (새 weight, 15개 지역). 시나리오마다 이어쓰기 저장 -> 중단 후 재실행 가능
done_c = set(pd.read_csv(out_cellwise)["scenario"]) if out_cellwise.exists() else set()
done_g = set(pd.read_csv(out_global)["scenario"]) if out_global.exists() else set()
for sc, n in zip(tqdm(scenarios), names):
    if n not in done_c:
        rows = {}
        for loc in regions:
            maxent_mat = make_maxent_mat(sc, loc)
            rows[loc] = [clu_SI(*make_CA_distribution_cellwise(maxent_mat, 77)) for _ in range(n_rep)]
        d = pd.DataFrame(rows); d.insert(0, "rep", range(n_rep)); d.insert(0, "scenario", n)
        d.to_csv(out_cellwise, mode="a", header=not out_cellwise.exists(), index=False)
    if n not in done_g:
        rows = {}
        for loc in regions:
            maxent_mat = make_maxent_mat(sc, loc)
            rows[loc] = [clu_SI(*make_CA_distribution_global(maxent_mat, 77, rep)) for rep in range(n_rep)]
        d = pd.DataFrame(rows); d.insert(0, "rep", range(n_rep)); d.insert(0, "scenario", n)
        d.to_csv(out_global, mode="a", header=not out_global.exists(), index=False)

  0%|          | 0/81 [00:00<?, ?it/s]

  1%|          | 1/81 [02:56<3:55:39, 176.74s/it]

  2%|▏         | 2/81 [05:52<3:52:07, 176.30s/it]

  4%|▎         | 3/81 [08:49<3:49:34, 176.59s/it]

  5%|▍         | 4/81 [11:42<3:44:42, 175.10s/it]

...

 96%|█████████▋| 78/81 [3:39:36<08:39, 173.25s/it]

 98%|█████████▊| 79/81 [3:42:24<05:43, 171.81s/it]

 99%|█████████▉| 80/81 [3:45:11<02:50, 170.37s/it]

100%|██████████| 81/81 [3:48:00<00:00, 169.81s/it]

100%|██████████| 81/81 [3:48:00<00:00, 168.89s/it]

In [12]:
# 19개(기존) + 15개(신규) = 34개 지역으로 합치기
old_c = pd.read_csv(res_dir / "77_lhs_repeat100_type1_newweight.csv")
old_g = pd.read_csv(res_dir / "77_lhs_repeat100_type2_newweight.csv")
new_c = pd.read_csv(out_cellwise)
new_g = pd.read_csv(out_global)
merged_c = old_c.merge(new_c, on=["scenario","rep"])
merged_g = old_g.merge(new_g, on=["scenario","rep"])
merged_c.to_csv(res_dir / "77_lhs_repeat100_type1_newweight_34regions.csv", index=False)
merged_g.to_csv(res_dir / "77_lhs_repeat100_type2_newweight_34regions.csv", index=False)
print(merged_c.shape, merged_g.shape)

(8100, 36) (8100, 36)
